# Lab 05 - Stage 0 And Stage A

Train small metadata and kinematic PCA-regression baselines.


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.decomposition import PCA
    from sklearn.linear_model import Ridge, Lasso
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.preprocessing import StandardScaler
except Exception as exc:
    raise RuntimeError("These labs require scikit-learn.") from exc


def make_synthetic_dataset(n=90, g=64, k=14, seed=11):
    """Return arrays with the same broad contract as the real pipeline."""
    rng = np.random.default_rng(seed)
    s_grid = np.linspace(0.0, 1.0, g)
    feature_names = [
        "knee_active_deg", "hip_active_deg", "elbow_active_deg",
        "trunk_vs_horizontal_deg", "spine_flexion_deg", "head_vs_trunk_deg",
        "knee_active_ddeg_ds", "hip_active_ddeg_ds", "elbow_active_ddeg_ds",
        "trunk_vs_horizontal_ddeg_ds", "spine_flexion_ddeg_ds",
        "head_vs_trunk_ddeg_ds", "handle_velocity_px_s", "handle_accel_px_s2",
    ][:k]

    seq = np.zeros((n, g, len(feature_names)), dtype=float)
    rate = rng.normal(27.0, 2.4, size=n)
    stroke_len = rng.normal(142.0, 8.0, size=n)
    drive_s = rng.normal(0.86, 0.08, size=n)
    athlete = np.where(np.arange(n) < n / 2, "athlete_A", "athlete_B")
    run = np.where(np.arange(n) % 3 == 0, "run_0", np.where(np.arange(n) % 3 == 1, "run_1", "run_2"))

    for i in range(n):
        phase = rng.normal(0, 0.025)
        knee = 55 + 92 * np.clip(s_grid + phase, 0, 1) + rng.normal(0, 2.0, g)
        hip = 42 + 38 * np.sin(0.5 * np.pi * np.clip(s_grid + 0.05 + phase, 0, 1)) + rng.normal(0, 1.5, g)
        elbow = 158 - 82 / (1 + np.exp(-14 * (s_grid - 0.67 + phase))) + rng.normal(0, 2.5, g)
        trunk = 72 + 28 * s_grid + rng.normal(0, 1.3, g)
        spine = 18 + 7 * np.sin(np.pi * s_grid) + rng.normal(0, 0.8, g)
        head = 9 + 2 * np.cos(2 * np.pi * s_grid) + rng.normal(0, 0.6, g)
        angles = [knee, hip, elbow, trunk, spine, head]
        cols = []
        for a in angles:
            cols.append(a)
        for a in angles:
            cols.append(np.gradient(a, s_grid))
        cols.append(460 * np.sin(np.pi * s_grid) + rng.normal(0, 10, g))
        cols.append(460 * np.pi * np.cos(np.pi * s_grid) + rng.normal(0, 25, g))
        seq[i] = np.stack(cols[:len(feature_names)], axis=1)

    curves = []
    for i in range(n):
        a = 2.45 + 0.05 * (rate[i] - rate.mean()) + rng.normal(0, 0.12)
        b = 3.65 + rng.normal(0, 0.18)
        shape = (s_grid ** a) * ((1 - s_grid) ** b)
        shape = shape / shape.max()
        coordination_bump = 0.04 * (seq[i, :, 0] - seq[i, :, 0].mean()) / max(seq[i, :, 0].std(), 1)
        peak = 505 + 4.0 * (stroke_len[i] - 140) + 7.0 * (rate[i] - 27) + rng.normal(0, 24)
        curve = peak * np.clip(shape + coordination_bump, 0, None) + rng.normal(0, 8, g)
        curves.append(np.maximum(0, curve))
    force_curves = np.asarray(curves)
    force_mask = np.ones_like(force_curves, dtype=bool)

    strokes = pd.DataFrame({
        "stroke_key": [f"synthetic__{i}" for i in range(n)],
        "run_name": run,
        "athlete_id": athlete,
        "match_seq_idx": np.arange(n),
        "stroke_rate_spm": rate,
        "stroke_length_cm": stroke_len,
        "rp3_drive_s": drive_s,
        "rp3_cycle_s": 60.0 / rate,
        "qc_excluded": False,
    })
    return {
        "strokes_df": strokes,
        "force_curves": force_curves,
        "force_mask": force_mask,
        "kinematic_sequences": seq,
        "s_grid": s_grid,
        "feature_names": feature_names,
    }


def load_or_synthetic_dataset():
    dataset_env = os.environ.get("DATASET_DIR", "").strip()
    dataset_dir = Path(dataset_env).expanduser() if dataset_env else None
    required = [
        "strokes.csv",
        "force_curves_resampled.npy",
        "kinematic_sequences.npy",
        "s_grid.npy",
        "feature_names.json",
    ]
    if dataset_dir is not None and dataset_dir.exists() and all((dataset_dir / name).exists() for name in required):
        print(f"Loading real dataset from {dataset_dir}")
        return {
            "strokes_df": pd.read_csv(dataset_dir / "strokes.csv"),
            "force_curves": np.load(dataset_dir / "force_curves_resampled.npy"),
            "force_mask": np.load(dataset_dir / "force_mask.npy") if (dataset_dir / "force_mask.npy").exists() else None,
            "kinematic_sequences": np.load(dataset_dir / "kinematic_sequences.npy"),
            "s_grid": np.load(dataset_dir / "s_grid.npy"),
            "feature_names": json.load(open(dataset_dir / "feature_names.json")),
        }
    if dataset_env:
        print(f"DATASET_DIR={dataset_dir} is missing required artifacts; using synthetic fallback data.")
    else:
        print("DATASET_DIR is not set; using synthetic fallback data.")
    return make_synthetic_dataset()


ds = load_or_synthetic_dataset()
strokes_df = ds["strokes_df"]
force_curves = ds["force_curves"]
kinematic_sequences = ds["kinematic_sequences"]
s_grid = ds["s_grid"]
feature_names = ds["feature_names"]
print("strokes_df:", strokes_df.shape)
print("force_curves:", force_curves.shape)
print("kinematic_sequences:", kinematic_sequences.shape)


In [ ]:
row_max = np.nanmax(force_curves, axis=1, keepdims=True)
valid = np.isfinite(force_curves).all(axis=1) & (row_max[:, 0] > 0)
Y_norm = force_curves[valid] / row_max[valid]
pca = PCA(n_components=min(5, Y_norm.shape[0], Y_norm.shape[1])).fit(Y_norm)
Y_scores = pca.transform(Y_norm)
work = strokes_df.loc[valid].reset_index(drop=True)
print('target scores shape:', Y_scores.shape)


In [ ]:
meta_cols = [c for c in ['stroke_rate_spm', 'stroke_length_cm', 'rp3_drive_s'] if c in work.columns]
X_meta = work[meta_cols].to_numpy(float)
scaler = StandardScaler().fit(X_meta)
model = Ridge(alpha=1.0).fit(scaler.transform(X_meta), Y_scores)
score_pred = model.predict(scaler.transform(X_meta))
curve_pred = pca.inverse_transform(score_pred) * row_max[valid]
rmse = np.sqrt(np.mean((force_curves[valid] - curve_pred) ** 2, axis=1))
print('Stage 0 metadata columns:', meta_cols)
print('in-sample median RMSE:', np.median(rmse))


In [ ]:
summary = {}
angle_cols = [c for c in feature_names if c.endswith('_deg')][:6]
for col in angle_cols:
    k = feature_names.index(col)
    vals = kinematic_sequences[valid, :, k]
    summary[f'{col}_min'] = np.nanmin(vals, axis=1)
    summary[f'{col}_max'] = np.nanmax(vals, axis=1)
    summary[f'{col}_range'] = np.nanmax(vals, axis=1) - np.nanmin(vals, axis=1)
    summary[f'{col}_mean'] = np.nanmean(vals, axis=1)
X_stageA_df = pd.concat([work[meta_cols].reset_index(drop=True), pd.DataFrame(summary)], axis=1)
X = X_stageA_df.to_numpy(float)
scaler_a = StandardScaler().fit(X)
models = {'ridge': Ridge(alpha=1.0), 'lasso': Lasso(alpha=0.001, max_iter=5000), 'gbr': MultiOutputRegressor(GradientBoostingRegressor(random_state=0))}
for name, m in models.items():
    m.fit(scaler_a.transform(X), Y_scores)
    pred = pca.inverse_transform(m.predict(scaler_a.transform(X))) * row_max[valid]
    print(name, 'median RMSE:', np.median(np.sqrt(np.mean((force_curves[valid] - pred) ** 2, axis=1))))
